# CIS 3400 Advanced Programming — Week 7
## Exploring & Visualizing Real Data with Python
### U.S. Airline On-Time Performance — June 2026

**Name:** Leland Smith  
**Date:** September 28, 2026

---

## Today's Goal

Today we will use **real U.S. airline flight records** to follow a complete Exploratory Data Analysis (EDA) workflow.

Our overall question is:

> **What can June 2026 flight data tell airline and airport managers about delays and operational patterns?**

We will work through:

**Question → Inspect → Clean → Transform → Group/Aggregate → Visualize → Interpret → Communicate**

### Individual Submission

Follow along during class, run the code, and complete every **STOP & ANSWER** section in your own words.

You will submit this completed notebook individually in Blackboard.

### AI / VIBE Coding

AI may help you understand code, debug, or explore an idea. However, **you are responsible for understanding the code and interpreting the results.**


# 1. Import Our Python Libraries

We will use:

- **pandas** — working with tables/DataFrames
- **matplotlib** — creating visualizations


In [6]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 70)


# 2. Load the Real BTS Flight Data

Upload the June 2026 BTS CSV to Colab, or place it in the same folder as this notebook if using Jupyter/VS Code.

Change the filename below **only if your downloaded file has a different name**.


In [7]:
FILE = "airline_ontime_2026.csv"

df = pd.read_csv(FILE, low_memory=False)

print("Data loaded!")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")


FileNotFoundError: [Errno 2] No such file or directory: 'airline_ontime_2026.csv'

## STOP & ANSWER 1 — Before We Analyze

A row in this dataset represents one scheduled flight.

Before looking closely at the results, what is **one pattern** you expect we might find in airline delays?

**Your prediction:**  
TYPE HERE


# 3. Inspect Before You Analyze


In [ ]:
display(df.head())


In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())


In [ ]:
df.info()


## STOP & ANSWER 2 — Know Your Data

1. How many rows are in the dataset?  
2. How many columns?  
3. Name **three columns** you think will be useful for understanding delays.  
4. Identify one column whose meaning you would need to investigate before using it.

**Your answers:**  
TYPE HERE


# 4. Select the Columns We Need

The original government dataset contains many fields. We do **not** need every field for every question.

Selecting useful columns makes our analysis easier to read and understand.

Notice: we keep the original `df` and create a smaller DataFrame called `flights`.


In [ ]:
useful_columns = [
    "FL_DATE",
    "DAY_OF_WEEK",
    "OP_CARRIER",
    "OP_CARRIER_FL_NUM",
    "ORIGIN",
    "ORIGIN_CITY_NAME",
    "DEST",
    "DEST_CITY_NAME",
    "CRS_DEP_TIME",
    "DEP_TIME",
    "DEP_DELAY",
    "DEP_DEL15",
    "DEP_TIME_BLK",
    "CRS_ARR_TIME",
    "ARR_TIME",
    "ARR_DELAY",
    "ARR_DEL15",
    "ARR_TIME_BLK",
    "CANCELLED",
    "CANCELLATION_CODE",
    "DIVERTED",
    "AIR_TIME",
    "DISTANCE",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

flights = df[useful_columns].copy()

print("Original shape:", df.shape)
print("Working shape:", flights.shape)
display(flights.head())


## STOP & ANSWER 3 — Why Make a Smaller DataFrame?

Why might it be useful to create `flights` instead of deleting unwanted columns from `df`?

**Your answer:**  
TYPE HERE


# 5. Data Quality: Missing Values


In [ ]:
missing = flights.isna().sum().sort_values(ascending=False)

display(
    pd.DataFrame({
        "Missing": missing,
        "Percent Missing": (missing / len(flights) * 100).round(1)
    }).head(15)
)


### Missing does NOT automatically mean bad data

Look at the delay-cause columns:

- `CARRIER_DELAY`
- `WEATHER_DELAY`
- `NAS_DELAY`
- `SECURITY_DELAY`
- `LATE_AIRCRAFT_DELAY`

A blank value may simply mean that the flight did not have a reportable delay of that type.

A diverted flight may also be missing a normal arrival delay because it did not complete the originally scheduled trip in the normal way.

**Data cleaning requires reasoning—not just deleting every row containing `NaN`.**


## STOP & ANSWER 4 — Think Like an Analyst

Choose **one column with missing values**.

Why might the value be missing? Would you automatically delete every row where that value is missing? Explain.

**Your answer:**  
TYPE HERE


# 6. Check for Duplicate Rows


In [ ]:
duplicate_count = flights.duplicated().sum()

print(f"Exact duplicate rows: {duplicate_count:,}")


## STOP & ANSWER 5

Why could duplicate flight records cause a problem when calculating counts or percentages?

**Your answer:**  
TYPE HERE


# 7. Fix the Date


In [ ]:
print("Before conversion:", flights["FL_DATE"].dtype)

flights["FL_DATE"] = pd.to_datetime(flights["FL_DATE"], errors="coerce")

print("After conversion:", flights["FL_DATE"].dtype)
print("First date:", flights["FL_DATE"].min())
print("Last date:", flights["FL_DATE"].max())


## STOP & ANSWER 6

Why is converting `FL_DATE` to a true datetime value useful?

**Your answer:**  
TYPE HERE


# 8. Understand Cancelled and Diverted Flights


In [ ]:
print("Cancelled flights:")
print(flights["CANCELLED"].value_counts(dropna=False))

print("\nDiverted flights:")
print(flights["DIVERTED"].value_counts(dropna=False))


In [ ]:
special = flights[
    (flights["CANCELLED"] == 1) | (flights["DIVERTED"] == 1)
]

display(
    special[
        ["FL_DATE", "OP_CARRIER", "ORIGIN", "DEST",
         "CANCELLED", "DIVERTED", "ARR_DELAY", "AIR_TIME"]
    ].head(10)
)


## STOP & ANSWER 7 — Important Cleaning Decision

Look at the cancelled/diverted examples.

Why might `ARR_DELAY` or `AIR_TIME` be missing for some of these flights?

Does that necessarily mean the data are incorrect?

**Your answer:**  
TYPE HERE


# 9. Create an Analysis DataFrame

For analyses specifically about **normal arrival delay**, we will exclude cancelled and diverted flights.

We are **not deleting them from the original data**. We are creating a new DataFrame appropriate for this particular question.


In [ ]:
completed = flights[
    (flights["CANCELLED"] == 0) &
    (flights["DIVERTED"] == 0) &
    (flights["ARR_DELAY"].notna())
].copy()

print(f"All scheduled flight records: {len(flights):,}")
print(f"Completed flights used for arrival-delay analysis: {len(completed):,}")


# 10. BTS Already Gives Us a Useful Variable

`ARR_DEL15` indicates whether a flight arrived **15 or more minutes late**.

- `0` = No
- `1` = Yes

Let's see how many completed flights were delayed.


In [ ]:
delay_counts = completed["ARR_DEL15"].value_counts().sort_index()
display(delay_counts)

delay_rate = completed["ARR_DEL15"].mean() * 100
print(f"Percent arriving 15+ minutes late: {delay_rate:.1f}%")


## STOP & ANSWER 8

Why is `ARR_DEL15` convenient for analysis compared with repeatedly writing a condition such as:

`ARR_DELAY >= 15`

**Your answer:**  
TYPE HERE


# 11. Grouping and Aggregation: Delay by Airline


In [ ]:
airline_summary = (
    completed.groupby("OP_CARRIER")
    .agg(
        Flights=("OP_CARRIER", "size"),
        AvgArrivalDelay=("ARR_DELAY", "mean"),
        MedianArrivalDelay=("ARR_DELAY", "median"),
        DelayRate=("ARR_DEL15", "mean")
    )
)

airline_summary["DelayRatePct"] = airline_summary["DelayRate"] * 100

airline_summary = airline_summary.sort_values("DelayRatePct", ascending=False)

display(airline_summary.round(2))


## STOP & ANSWER 9 — Mean vs. Median

Look at the average and median arrival delays.

Why can the **mean** and **median** tell different stories about flight delays?

**Your answer:**  
TYPE HERE


# 12. Visualize Airline Delay Rates


In [ ]:
(
    airline_summary
    .sort_values("DelayRatePct")["DelayRatePct"]
    .plot(
        kind="barh",
        figsize=(9, 6),
        title="Percent of Completed Flights Arriving 15+ Minutes Late — June 2026"
    )
)

plt.xlabel("Flights delayed 15+ minutes (%)")
plt.ylabel("Airline code")
plt.tight_layout()
plt.show()


## STOP & ANSWER 10 — Chart → Insight

Do **not** simply say "this chart shows airline delays."

Write 2–3 sentences explaining:

1. What pattern should a manager notice?
2. What does this chart **not** tell us or prove?

**Your answer:**  
TYPE HERE


# 13. What About Day of the Week?


In [ ]:
day_names = {
    1: "Monday",
    2: "Tuesday",
    3: "Wednesday",
    4: "Thursday",
    5: "Friday",
    6: "Saturday",
    7: "Sunday"
}

completed["DAY_NAME"] = completed["DAY_OF_WEEK"].map(day_names)

day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

day_summary = (
    completed.groupby("DAY_NAME")["ARR_DEL15"]
    .mean()
    .reindex(day_order) * 100
)

display(day_summary.round(1))


In [ ]:
day_summary.plot(
    kind="bar",
    figsize=(9, 5),
    title="Arrival Delay Rate by Day of Week — June 2026"
)

plt.ylabel("Flights delayed 15+ minutes (%)")
plt.xlabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## STOP & ANSWER 11

Which day has the highest delay rate? Which has the lowest?

Approximately how many **percentage points** separate them?

Would this result make you want to investigate further? Why?

**Your answer:**  
TYPE HERE


# 14. Does Time of Day Matter?


In [ ]:
time_summary = (
    completed.groupby("DEP_TIME_BLK")["ARR_DEL15"]
    .agg(["count", "mean"])
)

time_summary["DelayRatePct"] = time_summary["mean"] * 100

display(time_summary.round(2))


In [ ]:
time_summary["DelayRatePct"].plot(
    kind="line",
    marker="o",
    figsize=(11, 5),
    title="Arrival Delay Rate by Scheduled Departure Time — June 2026"
)

plt.ylabel("Flights delayed 15+ minutes (%)")
plt.xlabel("Scheduled departure time block")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## STOP & ANSWER 12 — Find the Story

Describe the overall pattern you see across the day.

Does the visualization suggest that delay risk stays constant throughout the day?

**Your answer:**  
TYPE HERE


# 15. Which Airports Are Busiest?


In [ ]:
origin_summary = (
    completed.groupby(["ORIGIN", "ORIGIN_CITY_NAME"])
    .agg(
        Flights=("ORIGIN", "size"),
        DelayRate=("ARR_DEL15", "mean")
    )
    .reset_index()
)

origin_summary["DelayRatePct"] = origin_summary["DelayRate"] * 100

busiest = origin_summary.sort_values("Flights", ascending=False).head(10)

display(busiest.round(2))


### Important analytical caution

A small airport with only a few flights could have an extreme percentage by chance.

When comparing groups, always consider **how many observations are in each group**.


# 16. Why Were Delayed Flights Delayed?


In [ ]:
delay_causes = [
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

cause_totals = completed[delay_causes].sum().sort_values(ascending=False)

display(cause_totals)


In [ ]:
cause_totals.rename({
    "CARRIER_DELAY": "Carrier",
    "WEATHER_DELAY": "Weather",
    "NAS_DELAY": "National Airspace System",
    "SECURITY_DELAY": "Security",
    "LATE_AIRCRAFT_DELAY": "Late Aircraft"
}).sort_values().plot(
    kind="barh",
    figsize=(9, 5),
    title="Recorded Delay Minutes by Cause — June 2026"
)

plt.xlabel("Total recorded delay minutes")
plt.ylabel("")
plt.tight_layout()
plt.show()


## STOP & ANSWER 13

Which recorded delay category accounts for the most total minutes in this dataset?

Why should we be careful about saying that this category **caused all airline delays**?

**Your answer:**  
TYPE HERE


# 17. Ask One Question of Your Own


Now use the dataset yourself.

Write **one additional question** that could be useful to an airline or airport manager.

Possible areas include:

- A particular airport
- A route
- Airline differences
- Distance
- Departure time
- Cancellation
- Delay cause

Do not simply repeat one of the questions we already answered.

### My Question

**TYPE YOUR QUESTION HERE**


In [ ]:
# YOUR ANALYSIS
#
# Write Python that helps answer your question.
# You may use Monday's examples and AI/VIBE coding for help.
# Be prepared to explain what your code does.


### What Did You Find?

Explain your result in **2–3 sentences in plain language**.

**Your interpretation:**  
TYPE HERE


# 18. Your 60-Second Managerial Summary


Imagine an airline or airport manager asks:

> **"What should I remember from this analysis?"**

Give them your **three most important findings** from today's work.

Do not talk about pandas, DataFrames, or Python. Talk about the **data and what it suggests**.

### Finding 1
TYPE HERE

### Finding 2
TYPE HERE

### Finding 3
TYPE HERE

### One thing management should investigate next
TYPE HERE


# 19. Exit Check


Complete these before submitting:

**One pandas command I understand better now:**  
TYPE HERE

**In my own words, `groupby()` lets us:**  
TYPE HERE

**One thing I learned about missing data:**  
TYPE HERE

**One thing that makes a visualization useful instead of merely attractive:**  
TYPE HERE

**One question I still have:**  
TYPE HERE

---

# Submit to Blackboard

1. Make sure your name is at the top.
2. Make sure every **STOP & ANSWER** section is complete.
3. Make sure your own analysis runs.
4. Save/download the notebook as an `.ipynb` file.
5. Submit it **individually** in Blackboard.

Next class, you and a partner will apply this same workflow to a **different real dataset** without having all of the analysis written for you.
